# NB6 - Capstone: A Self-Evolving, Auditable Agent

**Workshop: Self-Evolving Agents by Optimizing the Harness (no GPU)**

This is the payoff. Every component of **H = (E, T, C, S, L, V)** is now in play,
and we let the agent **evolve itself** - no human in the per-step loop.

- **EvoSkill** (evolutionary skill search): each generation breeds a small
  **population** of skill-library variants by *mutating* the current best -
  **adding** a skill distilled from a failure, or **dropping** a weak one - then
  **selects** the fittest on a held-out validation split. This is the outer
  **execution loop (E)** evolved into an optimizer.
- **ASG-SI** (auditable skill generation + self-improvement): every proposed
  change is scored with a **decomposed verifiable reward** (`score_sql`, our V)
  and written to an **audit log** with provenance - which generation, which
  operator, the measured val delta, and the accept/reject decision. Nothing enters
  the library (state **S**) without a recorded, replayable justification.

The result is a frozen LLM that gets measurably better **and** can explain exactly
*why* it changed - self-improvement you could put in front of a reviewer.

In [ ]:
import sys, os, json, re
sys.path.insert(0, os.path.abspath(".."))
from workshop_utils import (
    build_db, load_tasks, run_sql, score_sql, evaluate,
    llm, METER, SCHEMA_TEXT, extract_sql, baseline_prompt, make_agent,
    preflight, flush,
)
preflight("WANDB_API_KEY")    # OPENAI + LANGFUSE + W&B (see SETUP.md)
build_db()

train = [t for t in load_tasks() if t["split"] == "train"]
val   = train[::4]                          # held-out validation = fitness signal
opt   = [t for t in train if t not in val]  # where we mine failures to mutate from

try:
    baseline_acc = json.load(open("../data/baseline_test.json"))["accuracy"]
except FileNotFoundError:
    baseline_acc = evaluate(make_agent(), split="test")["accuracy"]
print(f"opt={len(opt)}  val={len(val)}  (test held out)")
print("baseline test accuracy:", round(baseline_acc, 3))

## The pieces: measure, mutate, audit

We reuse the SkillOpt machinery (forward pass + the reflection "gradient"), then
add the two evolutionary **mutation operators** and the **audit log** that makes
the run reviewable.

In [ ]:
def format_skills(skills):
    if not skills:
        return ""
    lines = ["Relevant skills (reusable SQL patterns the agent has learned):"]
    for s in skills:
        lines.append(f"- {s['name']}: USE WHEN {s['when_to_use']}. PATTERN: {s['pattern']}")
    return "\n".join(lines)

def evaluate_theta(theta, tasks):
    agent = make_agent(extra=format_skills(theta))
    fails, correct = [], 0
    for t in tasks:
        sql = agent(t["question"])
        if score_sql(sql, t["gold"]):
            correct += 1
        else:
            fails.append({"question": t["question"], "sql": sql, "gold": t["gold"]})
    return correct / len(tasks), fails

def val_fitness(theta):
    return evaluate_theta(theta, val)[0]

PROPOSE_SYS = (
    "You improve a text-to-SQL agent by writing ONE reusable SKILL that would have "
    "prevented a failure. Return STRICT JSON with keys name, when_to_use, pattern. "
    "Generalize to a CLASS of questions; never mention the specific question."
)

def parse_json_obj(raw):
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

def propose_skill(failure):                    # mutation: ADD (gradient from a failure)
    raw = llm([
        {"role": "system", "content": PROPOSE_SYS},
        {"role": "user", "content":
            "Schema:\n" + SCHEMA_TEXT + "\nQuestion: " + failure["question"] +
            "\n\nWrong SQL:\n" + failure["sql"] + "\n\nCorrect SQL:\n" + failure["gold"] +
            "\n\nWrite ONE general skill (JSON)."},
    ])
    d = parse_json_obj(raw)
    if not d or not all(k in d for k in ("name", "when_to_use", "pattern")):
        return None
    return {k: str(d[k]) for k in ("name", "when_to_use", "pattern")}

## The outer loop: a (1 + lambda) evolutionary strategy

Each **generation**:
1. Measure the current best library (`parent`) - its val **fitness** and its
   failures on the opt set.
2. Breed `lambda` **offspring** by mutation: ADD a skill distilled from a failure,
   or DROP an existing skill (to fight bloat).
3. **Select** the fittest of {parent, offspring...} on validation (**elitism** -
   the parent survives unless beaten), and record every trial in the audit log.

The decomposed verifiable reward (`val_fitness`) is the selection pressure; the
audit log is the receipt.

In [ ]:
import wandb

N_GEN      = 4       # generations
LAMBDA_ADD = 2       # ADD-mutations bred per generation

run = wandb.init(project="rl-agents-workshop", name="nb6-evoskill",
                 config={"model": "gpt-4o-mini", "generations": N_GEN, "lambda": LAMBDA_ADD})
wandb.define_metric("generation")
wandb.define_metric("val_fitness", step_metric="generation")

parent = []                 # the evolving skill library (state S)
audit  = []                 # ASG-SI: every proposed change, scored and decided
curve  = []
METER.reset()

for gen in range(N_GEN):
    pfit, pfails = evaluate_theta(parent, opt)
    pval = val_fitness(parent)
    curve.append((gen, pval, len(parent)))
    wandb.log({"generation": gen, "val_fitness": pval, "library_size": len(parent)})
    print(f"gen {gen}: val_fitness={pval:.2f}  library={len(parent)} skills")

    # --- breed offspring by mutation ------------------------------------------
    offspring = []                                  # (operator, new_theta, note)
    for f in pfails[:LAMBDA_ADD]:
        c = propose_skill(f)
        if c:
            offspring.append(("ADD", parent + [c], c["name"]))
    if parent:                                      # DROP-mutation: try removing one
        offspring.append(("DROP", parent[:-1], parent[-1]["name"]))

    # --- select the fittest (elitism: parent must be beaten) -------------------
    best_fit, best_theta, best_op, best_note = pval, parent, "ELITE", "-"
    for op, cand_theta, note in offspring:
        f = val_fitness(cand_theta)
        accepted = f > best_fit
        audit.append({"gen": gen, "op": op, "skill": note,
                      "val_fitness": round(f, 3), "decision": "accept" if accepted else "reject"})
        print(f"    {op:<4} {note:<26} val={f:.2f}  -> {'ACCEPT' if accepted else 'reject'}")
        if accepted:
            best_fit, best_theta, best_op, best_note = f, cand_theta, op, note
    parent = best_theta

print(f"\nevolved library: {len(parent)} skills, val_fitness={val_fitness(parent):.2f}")
print(METER)

In [ ]:
import matplotlib.pyplot as plt
xs = [c[0] for c in curve]
plt.figure(figsize=(6, 4))
plt.plot(xs, [c[1] for c in curve], marker="o", color="seagreen", label="val fitness")
plt.axhline(baseline_acc, ls="--", color="gray", label="NB0 baseline (test)")
plt.xlabel("generation"); plt.ylabel("validation fitness"); plt.ylim(0, 1)
plt.title("EvoSkill: the library evolves itself (weights frozen)")
plt.legend(); plt.show()

## The audit trail (ASG-SI)

Self-improvement you can review: every change the agent *considered*, the
**measured** validation reward, and the accept/reject decision. Nothing entered
the library without this receipt.

In [ ]:
print(f"{'gen':>3}  {'operator':<6} {'skill':<26} {'val':>5}  decision")
print("-" * 60)
for a in audit:
    print(f"{a['gen']:>3}  {a['op']:<6} {a['skill']:<26} {a['val_fitness']:>5}  {a['decision']}")

print("\nFinal library (provenance = the audit rows above that were accepted):")
for s in parent:
    print(f"  - {s['name']}: USE WHEN {s['when_to_use'][:64]}")

## The capstone number

Report the evolved library on the **held-out test split** - the number we never
optimized against - and place it next to the NB0 baseline.

In [ ]:
METER.reset()
final_test = evaluate(make_agent(extra=format_skills(parent)), split="test")
acc = final_test["accuracy"]
print("baseline (NB0, no skills)      :", round(baseline_acc, 3))
print("self-evolved agent (NB6)       :", round(acc, 3))
print("by level:", {k: round(v["acc"], 2) for k, v in final_test["by_level"].items()})
print(METER)

wandb.summary["baseline_acc"]   = baseline_acc
wandb.summary["final_test_acc"] = acc
wandb.summary["library_size"]   = len(parent)
wandb.summary["audit_events"]   = len(audit)
wandb.finish()
flush()

## Takeaways - the whole thesis, realized

- We built an agent (NB0), measured it (NB1, **V**), gave it memory (NB2, **S**),
  structured that into a gated skill library (NB3), **trained** it like a net
  (NB4), made it **hierarchical and transferable** (NB5), and finally let it
  **evolve itself with an audit trail** (NB6) - all with the **weights frozen**.
- **Reflection was the gradient, the skill document was the parameter vector, and
  the eval set was the loss.** Self-evolution = optimizing the **harness**, not the
  brain. No GPU, on a laptop, for cents.
- **Auditable** beats merely "better": the ASG-SI log means every gain is
  traceable to a measured reward and a recorded decision - the difference between a
  demo and something you would actually ship.

### Where to take it next
1. Replace the `(1 + lambda)` strategy with a real **population** (crossover
   between two libraries). Does diversity find skills a greedy search misses?
2. Add a **regression test**: re-score accepted skills each generation and DROP any
   whose contribution has decayed. (Skills can go stale as the library changes.)
3. Swap the toy DB for your own schema and tasks. The harness is the product - the
   database is just where you point it.